In [1]:
# =========================================================
# NOTEBOOK: 04_feature_engineering.ipynb
# =========================================================


# =========================================================
# STEP 1 — IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# =========================================================
# STEP 2 — LOAD MASTER DATASET
# =========================================================

master_df = pd.read_csv(
    r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\processed\master_dataset.csv'
)

print("Master Dataset Loaded Successfully")

Master Dataset Loaded Successfully


In [3]:

# =========================================================
# STEP 3 — DATE CONVERSION
# =========================================================

date_cols = [

    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_creation_date',
    'review_answer_timestamp'
]

for col in date_cols:

    master_df[col] = pd.to_datetime(
        master_df[col],
        errors='coerce'
    )

print("Date Conversion Completed")

Date Conversion Completed


In [4]:
# =========================================================
# STEP 4 — BASIC FEATURE ENGINEERING
# =========================================================

# ---------------------------------------------------------
# DELIVERY FEATURES
# ---------------------------------------------------------

master_df['delivery_duration_days'] = (

    master_df['order_delivered_customer_date']
    - master_df['order_purchase_timestamp']

).dt.days


master_df['delivery_delay_days'] = (

    master_df['order_delivered_customer_date']
    - master_df['order_estimated_delivery_date']

).dt.days


master_df['approval_time_hours'] = (

    master_df['order_approved_at']
    - master_df['order_purchase_timestamp']

).dt.total_seconds() / 3600


# ---------------------------------------------------------
# PURCHASE TIME FEATURES
# ---------------------------------------------------------

master_df['purchase_year'] = (
    master_df['order_purchase_timestamp'].dt.year
)

master_df['purchase_month'] = (
    master_df['order_purchase_timestamp'].dt.month
)

master_df['purchase_day'] = (
    master_df['order_purchase_timestamp'].dt.day
)

master_df['purchase_hour'] = (
    master_df['order_purchase_timestamp'].dt.hour
)

master_df['purchase_weekday'] = (
    master_df['order_purchase_timestamp'].dt.day_name()
)

master_df['purchase_weekend'] = np.where(

    master_df['purchase_weekday'].isin(
        ['Saturday', 'Sunday']
    ),

    1,
    0
)

print("Basic Features Created Successfully")

Basic Features Created Successfully


In [5]:
# =========================================================
# STEP 5 — CUSTOMER-LEVEL FEATURES
# =========================================================

customer_features = (

    master_df.groupby('customer_unique_id')

    .agg({

        'order_id': 'nunique',

        'payment_value': ['sum', 'mean'],

        'review_score': 'mean',

        'delivery_delay_days': 'mean',

        'product_id': 'nunique'
    })

)

customer_features.columns = [

    'total_orders',
    'total_spent',
    'avg_order_value',
    'avg_review_score',
    'avg_delivery_delay',
    'unique_products_purchased'
]

customer_features = customer_features.reset_index()

print("Customer-Level Features Created")

Customer-Level Features Created


In [6]:
# =========================================================
# STEP 6 — RFM FEATURES
# =========================================================

snapshot_date = (
    master_df['order_purchase_timestamp'].max()
)

rfm = (

    master_df.groupby('customer_unique_id')

    .agg({

        'order_purchase_timestamp': lambda x: (
            snapshot_date - x.max()
        ).days,

        'order_id': 'nunique',

        'payment_value': 'sum'

    })

    .reset_index()
)

rfm.columns = [

    'customer_unique_id',
    'Recency',
    'Frequency',
    'Monetary'
]

print("RFM Features Created")

RFM Features Created


In [7]:
# =========================================================
# STEP 7 — CHURN FEATURE ENGINEERING
# =========================================================

# ---------------------------------------------------------
# CHURN LABEL
# ---------------------------------------------------------

# Example logic:
# if customer inactive for > 90 days → churned

rfm['churn'] = np.where(

    rfm['Recency'] > 90,

    1,
    0
)

print("Churn Labels Created")

Churn Labels Created


In [8]:
# =========================================================
# STEP 8 — PRODUCT FEATURES
# =========================================================

product_features = (

    master_df.groupby('product_id')

    .agg({

        'payment_value': 'sum',

        'order_id': 'count',

        'review_score': 'mean',

        'delivery_delay_days': 'mean'

    })

)

product_features.columns = [

    'product_revenue',
    'product_total_orders',
    'product_avg_review',
    'product_avg_delay'
]

product_features = product_features.reset_index()

print("Product Features Created")

Product Features Created


In [9]:
# =========================================================
# STEP 9 — SELLER FEATURES
# =========================================================

seller_features = (

    master_df.groupby('seller_id')

    .agg({

        'payment_value': 'sum',

        'order_id': 'count',

        'review_score': 'mean',

        'delivery_delay_days': 'mean'

    })

)

seller_features.columns = [

    'seller_total_revenue',
    'seller_total_orders',
    'seller_avg_review',
    'seller_avg_delay'
]

seller_features = seller_features.reset_index()

print("Seller Features Created")

Seller Features Created


In [11]:

# =========================================================
# STEP 10 — PAYMENT FEATURES
# =========================================================

payment_features = pd.get_dummies(

    master_df['payment_type'],
    prefix='payment'
)

master_df = pd.concat(
    [master_df, payment_features],
    axis=1
)

print("Payment Features Created")

Payment Features Created


In [12]:
# =========================================================
# STEP 11 — REVIEW FEATURES
# =========================================================

master_df['review_length'] = (

    master_df['review_comment_message']
    .astype(str)
    .apply(len)
)

master_df['review_word_count'] = (

    master_df['review_comment_message']
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print("Review Features Created")

Review Features Created


In [13]:
# =========================================================
# STEP 12 — CATEGORY ENCODING
# =========================================================

label_cols = [

    'customer_gender',
    'customer_segment',
    'product_category_name',
    'product_brand',
    'payment_type',
    'purchase_weekday'
]

le = LabelEncoder()

for col in label_cols:

    master_df[col] = le.fit_transform(
        master_df[col].astype(str)
    )

print("Label Encoding Completed")


Label Encoding Completed


In [14]:
# =========================================================
# STEP 13 — MERGE ENGINEERED FEATURES
# =========================================================

master_df = master_df.merge(

    customer_features,
    on='customer_unique_id',
    how='left'
)

master_df = master_df.merge(

    rfm,
    on='customer_unique_id',
    how='left'
)

master_df = master_df.merge(

    product_features,
    on='product_id',
    how='left'
)

master_df = master_df.merge(

    seller_features,
    on='seller_id',
    how='left'
)

print("Feature Merging Completed")

Feature Merging Completed


In [15]:
# =========================================================
# STEP 14 — HANDLE MISSING VALUES
# =========================================================

numerical_cols = master_df.select_dtypes(
    include=np.number
).columns

for col in numerical_cols:

    master_df[col] = master_df[col].fillna(
        master_df[col].median()
    )

categorical_cols = master_df.select_dtypes(
    include='object'
).columns

for col in categorical_cols:

    master_df[col] = master_df[col].fillna(
        'Unknown'
    )

print("Missing Values Handled")

Missing Values Handled


In [16]:
# =========================================================
# STEP 15 — FEATURE SCALING
# =========================================================

scale_cols = [

    'payment_value',
    'delivery_duration_days',
    'delivery_delay_days',
    'approval_time_hours',
    'review_length',
    'review_word_count',
    'total_spent',
    'avg_order_value',
    'Recency',
    'Frequency',
    'Monetary'
]

scaler = StandardScaler()

master_df[scale_cols] = scaler.fit_transform(
    master_df[scale_cols]
)

print("Feature Scaling Completed")

Feature Scaling Completed


In [17]:
# =========================================================
# STEP 16 — FEATURE SELECTION
# =========================================================

drop_cols = [

    'customer_name',
    'seller_company_name',
    'seller_contact_name',
    'review_comment_title',
    'review_comment_message'
]

master_df.drop(
    columns=drop_cols,
    inplace=True,
    errors='ignore'
)

print("Unnecessary Columns Removed")

Unnecessary Columns Removed


In [18]:
# =========================================================
# STEP 17 — FINAL FEATURE DATASET
# =========================================================

print("\nFinal Feature Engineered Dataset Shape:")
print(master_df.shape)

print("\nFeature Engineered Dataset Sample:\n")

display(master_df.head())


Final Feature Engineered Dataset Shape:
(2530433, 87)

Feature Engineered Dataset Sample:



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_duration_days,delivery_delay_days,approval_time_hours,purchase_month,purchase_year,purchase_weekday,customer_unique_id,customer_gender,customer_age,customer_zip_code_prefix,customer_city,customer_state,customer_segment,order_item_id,product_id,seller_id,shipping_limit_date,price_x,freight_value,discount_rate,product_category_name,product_name,product_brand,product_weight_g,product_length_cm,product_height_cm,product_width_cm,cost,price_y,seller_contact_gender,seller_contact_age,seller_zip_code_prefix,seller_city,seller_state,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_creation_date,review_answer_timestamp,purchase_day,purchase_hour,purchase_weekend,payment_apple_pay,payment_bank_transfer,payment_boleto,payment_credit_card,payment_debit_card,payment_paypal,payment_voucher,payment_apple_pay,payment_bank_transfer,payment_boleto,payment_credit_card,payment_debit_card,payment_paypal,payment_voucher,review_length,review_word_count,total_orders,total_spent,avg_order_value,avg_review_score,avg_delivery_delay,unique_products_purchased,Recency,Frequency,Monetary,churn,product_revenue,product_total_orders,product_avg_review,product_avg_delay,seller_total_revenue,seller_total_orders,seller_avg_review,seller_avg_delay
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,0.037357,0.036951,-1.282531,12,2025,2,d45cdff2-5195-41e2-a0e5-6fe597e378dd,1,47,85037,Phoenix,AZ,0,1,89804d82-33a1-4558-8fa2-9b0252a2a406,36c2b043-ccf4-4d9c-a0bb-4b8d28737fd7,2025-12-30 07:07:20,916.90,45.47,0.0,4,Nova Desk - Industrial,3,24130,108,10,13,395.24,916.90,M,18,90086,Los Angeles,CA,1,5,1,0.725549,Unknown,4.0,NaT,NaT,27,7,1,False,False,False,False,False,True,False,False,False,False,False,False,True,False,-3.381258,-3.282810,7,-0.233903,-0.078063,3.222222,0.333333,12,-0.756465,0.911663,-0.233903,0,1234480.37,724,3.800000,-8.556017,6655560.54,5121,3.952081,-9.193164
1,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,0.037357,0.036951,-1.282531,12,2025,2,d45cdff2-5195-41e2-a0e5-6fe597e378dd,1,47,85037,Phoenix,AZ,0,2,ac10b78c-342e-4b50-9ea9-bddde7db2e79,09936504-415b-4f90-a5b8-ad15a5be67d0,2025-12-30 07:07:20,817.14,172.48,0.0,2,Vertex Smartwatch S468,6,586,28,19,23,717.81,817.14,M,70,19137,Philadelphia,PA,1,5,1,0.725549,Unknown,4.0,NaT,NaT,27,7,1,False,False,False,False,False,True,False,False,False,False,False,False,True,False,-3.381258,-3.282810,7,-0.233903,-0.078063,3.222222,0.333333,12,-0.756465,0.911663,-0.233903,0,4597968.11,2719,3.779849,-16.312730,6574599.35,4999,3.996998,-18.445178
2,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,0.037357,0.036951,-1.282531,12,2025,2,d45cdff2-5195-41e2-a0e5-6fe597e378dd,1,47,85037,Phoenix,AZ,0,3,f18e6310-0e75-4b7f-bc20-90ac1e7bd466,7d336012-7101-4a66-b6bc-c269620d8df9,2025-12-30 07:07:20,37.04,94.98,0.0,3,Crest Minimal Dress,1,697,55,7,7,9.68,37.04,F,27,98129,Seattle,WA,1,5,1,0.725549,Unknown,4.0,NaT,NaT,27,7,1,False,False,False,False,False,True,False,False,False,False,False,False,True,False,-3.381258,-3.282810,7,-0.233903,-0.078063,3.222222,0.333333,12,-0.756465,0.911663,-0.233903,0,1801075.20,2017,4.059165,-3.545635,6634183.36,4946,3.950022,-18.226289
3,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,0.033655,0.036951,1.601462,6,2019,0,35cee471-325e-4ad2-8e4e-7b169dc6df81,1,18,90060,Los Angeles,CA,0,1,5816f107-b725-4c41-b794-298bf9669a41,91fe2cc8-51a5-4c79-9c12-cea

In [19]:
# =========================================================
# STEP 18 — SAVE FEATURE ENGINEERED DATASET
# =========================================================

master_df.to_csv(

    r'C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\processed\feature_engineered_dataset.csv',

    index=False
)

print("\nFeature Engineered Dataset Saved Successfully")


Feature Engineered Dataset Saved Successfully


In [20]:
# =========================================================
# STEP 19 — FEATURE ENGINEERING SUMMARY
# =========================================================

print("\n===================================================")
print("FEATURE ENGINEERING SUMMARY")
print("===================================================")

print("""

Features Created:

✔ Delivery Features
✔ Purchase Time Features
✔ Customer Features
✔ Product Features
✔ Seller Features
✔ Payment Features
✔ Review Text Features
✔ RFM Features
✔ Churn Labels

Preprocessing Done:

✔ Encoding
✔ Scaling
✔ Missing Value Handling
✔ Feature Merging
✔ Feature Selection

Output File:

data/processed/feature_engineered_dataset.csv

""")


FEATURE ENGINEERING SUMMARY


Features Created:

✔ Delivery Features
✔ Purchase Time Features
✔ Customer Features
✔ Product Features
✔ Seller Features
✔ Payment Features
✔ Review Text Features
✔ RFM Features
✔ Churn Labels

Preprocessing Done:

✔ Encoding
✔ Scaling
✔ Missing Value Handling
✔ Feature Merging
✔ Feature Selection

Output File:

data/processed/feature_engineered_dataset.csv




In [21]:

# =========================================================
# STEP 20 — NOTEBOOK COMPLETION
# =========================================================

print("\n===================================================")
print("04_feature_engineering.ipynb COMPLETED SUCCESSFULLY")
print("===================================================")


04_feature_engineering.ipynb COMPLETED SUCCESSFULLY
